In [2]:
import cv2
import numpy as np
import math
import os

In [3]:
def save_image_to_mem(image_path, output_file, act_size=256):
    # 1. Preprocess
    original_img = cv2.imread(image_path)
    input_img = cv2.resize(original_img, (act_size, act_size))
    rgb_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB) # Shape: (H, W, C)

    # 2. Convert uint8 [0,255] to int8 [-128, 127]
    # We cast to int16 first to avoid wrap-around during subtraction
    int8_img = (rgb_img.astype(np.int16) - 128).astype(np.int8)

    # 3. Reorder to [Channel, Height, Width]
    ch_first = int8_img.transpose(2, 0, 1) 

    with open(output_file, 'w') as f:
        for c in range(ch_first.shape[0]):
            flat_channel = ch_first[c].flatten()
            for val in flat_channel:
                # Convert to 2-digit 2's complement hex
                hex_val = format(val & 0xFF, '02x')
                f.write(f"{hex_val}\n")

In [4]:
def save_weights_to_mem(weights_decimal_path, output_file, out_ch_idx=0, c_in=3, k=3):
    # 1. Load signed decimals from file
    with open(weights_decimal_path, 'r') as f:
        # Assumes file has one decimal per line or space-separated
        all_weights = [int(x) for x in f.read().split()]
    
    # 2. Slice for the specific output channel
    # Shape: [out_ch, kH, kW, in_ch]
    weights_per_out_ch = k * k * c_in
    start = out_ch_idx * weights_per_out_ch
    end = start + weights_per_out_ch
    current_weights = all_weights[start:end]

    # 3. Write to .mem as hex
    with open(output_file, 'w') as f:
        for val in current_weights:
            # Convert to 2-digit 2's complement hex
            hex_val = format(val & 0xFF, '02x')
            f.write(f"{hex_val}\n")

In [11]:
def write_scalar_mem(input_txt, output_mem, bits):
    """
    Reads a signed decimal from a .txt file, ignoring lines starting with '//',
    and writes its two's complement hex representation to a .mem file.
    """
    try:
        with open(input_txt, 'r') as f:
            lines = f.readlines()
            
        # Filter out empty lines and lines starting with '//'
        valid_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped and not stripped.startswith("//"):
                valid_lines.append(stripped)
                
        if not valid_lines:
            print(f"Warning: {input_txt} contains no valid integer data.")
            return
            
        # Assume the first non-comment, non-empty line contains the value
        value = int(valid_lines[0])

        # 1. Convert to Two's Complement Unsigned Equivalent
        mask = (1 << bits) - 1
        unsigned_val = value & mask

        # 2. Format as Hex
        chars = math.ceil(bits / 4)
        hex_val = format(unsigned_val, f'0{chars}x')

        # 3. Write to .mem file
        with open(output_mem, 'w') as f:
            f.write(hex_val + "\n")
            
        print(f"Converted {input_txt} ({value}) -> {output_mem} ({hex_val})")

    except FileNotFoundError:
        print(f"Error: {input_txt} not found.")
    except ValueError:
        print(f"Error: Could not parse integer from the valid data in {input_txt}.")

In [ ]:
import os
from pathlib import Path

_HERE = Path(os.path.abspath("")).resolve() if "__file__" not in dir() else Path(__file__).resolve().parent
_PROJECT = _HERE.parent.parent

OUTPUT_DIR = str(_PROJECT / "hardware" / "testbench" / "mem")
IMAGE_PATH = str(_PROJECT / "software" / "inference" / "data" / "input_image.jpg")
LAYER_DIR = str(_PROJECT / "hardware" / "weights" / "layers" / "layer_00")

# Create output directory if it doesn't exist
try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Directory '{OUTPUT_DIR}' ensured to exist.")
except OSError as e:
    # Handle other potential errors like permission issues
    print(f"Error creating directory {OUTPUT_DIR}: {e}")

# convert image
save_image_to_mem(IMAGE_PATH, f"{OUTPUT_DIR}/pixels.mem")

# convert layer00 parameters
save_weights_to_mem(f"{LAYER_DIR}/weights.hex", f"{OUTPUT_DIR}/weights.mem", out_ch_idx=0, c_in=3, k=3)
write_scalar_mem(f"{LAYER_DIR}/bias.hex", f"{OUTPUT_DIR}/bias.mem", bits=32)
write_scalar_mem(f"{LAYER_DIR}/m0.txt", f"{OUTPUT_DIR}/m0.mem", bits=32)
write_scalar_mem(f"{LAYER_DIR}/n_shift.txt", f"{OUTPUT_DIR}/n_shift.mem", bits=6)
write_scalar_mem(f"{LAYER_DIR}/zp_in.txt", f"{OUTPUT_DIR}/zp_in.mem", bits=8)
write_scalar_mem(f"{LAYER_DIR}/zp_out.txt", f"{OUTPUT_DIR}/zp_out.mem", bits=8)